huggingface 라이브러리를 사용하여 embedding 추출하는 것을 목표로 함

huggingface LLM Course documentation을 적극 참고하는 중..

**갑자기 생각난건데 d_model 즉 히든dim를 4개로만 하면 4가지 속성에 대해서만 탐지하니까 걍 분류가 되지 않을까? 라는 상상을 함


모델 사용을 가장 쉽게 하기 위해선 pipeline() 함수를 활용할 수 있다.

전처리/후처리 과정을 모델과 연결시켜주는 역할!

즉, 원하는 모델 / 용도를 설정하면, 별도의 전처리/후처리 없이 바로 결과를 얻어낼 수 있게 한다.

굳이 따지면 우리 프로젝트는 후처리에서 분류 모델?을? 만든다고 해야 할까?


https://huggingface.co/docs/transformers/main/ko/tasks/zero_shot_image_classification


제로샷이라는게 파이프라인에 있는데 이건 라벨 정답 데이터가 없어도 레이블 달고 분류를 할 수 있다네? 어떻게???

아무튼 목표는 Multi-label Classification 임

근데 그럴려면 소프트맥스 말고 시그모이드를 쓰는게 나을거같음

 소프트맥스는 마지막 전체 클래스의 합이 1

 시그모이드는 그 부분에만 확률 0-1

 ReLU >??

 오차 계산(손실함수)도 클래스 개별로 봐야하니 bnary cross entropy

Cross-Entropy > Focal Loss

MSE

Log Loss

경사하강


 결정 경게(thresholding)도 그냥 0.5말고 캘리브레이션 해줘야 > MCC Threshold Optimization

In [1]:
!pip install -q transformers tqdm torch numpy pandas

from transformers import AutoModelForMaskedLM #AutoTokenizer

import os
import tqdm
import torch
import numpy as np
import pandas as pd

In [2]:
#from huggingface_hub import login
#login()

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')
#SAVE_PATH = '/content/drive/MyDrive/Github/Mprotein_hydrophobic/ESMC_embedding'
#os.makedirs(SAVE_PATH, exist_ok=True)

In [2]:
save_path = './ESMC_embedding'
os.makedirs(save_path, exist_ok=True)

In [3]:
model_id = 'Synthyra/ESMplusplus_large'
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


2. [h5py] 파일 열기('w') -> 빈 데이터셋 4개 생성 (embeddings, mask, labels, fold_ids)


3. [Loop] for 배치 in 데이터프레임:
    a. [Tokenizer] 시퀀스 -> 토큰화 (길이 1024 고정, Padding)
    b. [Model] 토큰 -> 모델 -> last_hidden_state (3D 벡터) 추출
    c. [Numpy] GPU 텐서 -> CPU 넘파이 변환 (float16)
    d. [h5py] 데이터셋 resize -> 데이터 밀어넣기

In [75]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [76]:
!nvidia-smi
'''AttributeError
try:
    del model
    del output
    del tokenized
except NameError:
    pass'''

Fri Jan 16 05:23:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             34W /   70W |    3872MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

'AttributeError\ntry:\n    del model\n    del output\n    del tokenized\nexcept NameError:\n    pass'

In [27]:
#모델/토크나이저 호출
model = AutoModelForMaskedLM.from_pretrained(model_id, trust_remote_code=True)
model = model.to(device).eval()
tokenizer = model.tokenizer

In [90]:
#csv에서 데이터 호출, 시퀀스 길이가 1500자 넘으면 그 행 drop
MAX_LEN = 1500
df = pd.read_csv('https://github.com/Rainbowbarkbark/Mprotein_hydrophobic/blob/main/Swissprot_Membrane_Train_Validation_dataset.csv?raw=true')
df = df[df['Sequence'].str.len() <= MAX_LEN]
df = df.reset_index(drop=True)
labels = ['Peripheral', 'Transmembrane', 'LipidAnchor', 'Soluble']
df_labels = df[labels].values.astype('int8')
df_partitions = df['Partition'].values.astype('int8')
df_sequences_raw = df['Sequence'].values.tolist()

In [86]:
print(df['Sequence'].map(len).max())
print(len(df_sequences_raw))


1500
26991


In [ ]:
#임베딩 추출 함수
def embedding_extract(model, tokenizer, per_sequence, device, max_length):
    tokenized = tokenizer(
        per_sequence, 
        padding = True, 
        return_tensors ='pt', 
        max_length = max_length)
    attention_mask = tokenized['attention_mask']
    tokenized = {key: val.to(device) for key, val in tokenized.items()}
    with torch.no_grad():
        output = model(**tokenized, output_hidden_states=True)
    embeddings = output.last_hidden_state.cpu().numpy()
    attention_mask = attention_mask.cpu().numpy()

    return embeddings, attention_mask

In [ ]:
embedding_extract(model, tokenizer, df_sequences_raw[0:2], device, MAX_LEN)

In [108]:
#저장할 파일 만들기
import h5py

#1.저장할 파일명
filename = 'ESMC_embedding.h5'
FILE_PATH = os.path.join(save_path, filename)
BATCH_SIZE = 10


#2. partition, label 정보는 바로 저장, 어텐션 마스크와 임베딩은 빈 파일로 생성
with h5py.File(FILE_PATH, 'w') as f:
    f.create_dataset('partitions', data=df_partitions)
    f.create_dataset('labels', data=df_labels)
    f.create_dataset('embeddings',
                     shape=(0, MAX_LEN, 1152),
                     maxshape = (None, MAX_LEN, 1152),
                     chunks = True,
                     dtype='float32')
    f.create_dataset('attention_masks',
                     shape=(0, MAX_LEN),
                     maxshape=(None, MAX_LEN),
                     chunks=True,
                     dtype='int8')

In [ ]:
for i in tqdm(range(0, len(df_sequences_raw), BATCH_SIZE)):
    batch_sequences = df_sequences_raw[i:i+BATCH_SIZE]
    embeddings, attention_masks = embedding_extract(model, tokenizer, batch_sequences, device, MAX_LEN)
    
    with h5py.File(FILE_PATH, 'a') as f:
        #현재 데이터셋 크기
        curr_size = f['embeddings'].shape[0]
        
        #데이터셋 크기 늘리기
        f['embeddings'].resize(curr_size + BATCH_SIZE, axis=0)
        f['attention_masks'].resize(curr_size + BATCH_SIZE, axis=0)
        
        #새로운 데이터 추가
        f['embeddings'][curr_size:] = embeddings.astype('float32')
 
        f['attention_masks'][curr_size:] = attention_masks.astype('int8')

  4%|▎         | 97/2700 [13:03<5:33:33,  7.69s/it]

In [ ]:
embedding_dict = model.embed_dataset(
    sequences=[
        'MALWMRLLPLLALLALWGPDPAAA', ... # list of protein sequences
    ],
    batch_size=10, # adjust for your GPU memory
    max_len=512, # adjust for your needs
    full_embeddings=False, # if True, no pooling is performed
    embed_dtype=torch.float32, # cast to what dtype you want
    pooling_type=['mean', 'cls'], # more than one pooling type will be concatenated together
    num_workers=0, # if you have many cpu cores, we find that num_workers = 4 is fast for large datasets
    sql=False, # if True, embeddings will be stored in SQLite database
    sql_db_path='embeddings.db',
    save=True, # if True, embeddings will be saved as a .pth file
    save_path='embeddings.pth',
)
# embedding_dict is a dictionary mapping sequences to their embeddings as tensors for .pth or numpy arrays for sql